# Magnitude Composition and Monte Carlo Precision

This notebook calls two separate components: a candidate combination of supplied one-sided difference/sum p-values, and deterministic binomial precision planning with an independent R calculation. Software checks do not establish the validity of the constituent tests or calibrate the magnitude procedure. No model is fitted, no response is simulated and no recruitment target is selected.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if sys.version_info[:3] != (3, 10, 9):
    raise RuntimeError("Select the reference Python 3.10.9 kernel before installation.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT),
                "numpy==1.23.5", "scipy==1.10.0", "pandas==2.0.3", "matplotlib==3.7.5"], check=True)
sys.path.insert(0, str(ROOT))
from src.tutoring_statistics.magnitude_composition import COMPARISON_IDS, compose_family
# Precision readback also requires R 4.2.2 with jsonlite 1.8.8; see the folder README.


## Candidate Magnitude Test

For each recipient and comparison, let a be the C1 slope and b the reference slope, using initial-need amounts. With D = a - b and S = a + b, abs(a) < abs(b) means D*S < 0. Two prespecified intersection-union branches are combined by Bonferroni2, followed by Holm across the four fixed comparisons. No tail is selected from an observed sign.

The following values are artificial interface fixtures, not fitted p-values or empirical findings. An actual input requires each contrast's own standard error and degrees of freedom. Statistical unavailability retains a null reported p-value and pads only the internal Holm calculation with 1; technical uncertainty remains unresolved.

In [ ]:
RUN_COMPOSITION = False
illustrative_records = [
    {"comparisonID": comparison, "status": "available", "reason": None,
     "p": {"D_less": p / 2, "D_greater": 1, "S_less": 1, "S_greater": p / 2}}
    for comparison, p in zip(COMPARISON_IDS, (0.008, 0.016, 0.03, 0.08))
]
if RUN_COMPOSITION:
    display(compose_family(illustrative_records))


## Precision Planning

The fixed planning rule asks for the smallest integer B such that, for K ~ Binomial(B, 0.05), the probability of a central 95% Clopper-Pearson interval having half-width at most 0.01 is at least 0.90. Every B from 1 and every count 0..B is evaluated; numerical ambiguity is an error, not rounded away.

The reference calculation selected B = 2,129 with assurance about 0.90120. That is a Monte Carlo precision result, not a demonstrated Type-I error rate, a simultaneous guarantee, a power result or a human sample size. Running the cells below recomputes the rule; the reference answer is not supplied as a calculation input. Each output directory must be new.

In [ ]:
RUN_PRECISION = False
RUN_INDEPENDENT_READBACK = False
OUTPUT = ROOT / "results/magnitude_precision"
INDEPENDENT_OUTPUT = ROOT / "results/magnitude_precision_independent"
RSCRIPT = "Rscript"
producer_command = [sys.executable, str(ROOT / "scripts/plan_monte_carlo_precision.py"),
                    "--output", str(OUTPUT)]
reader_command = [RSCRIPT, "--vanilla", str(ROOT / "scripts/read_monte_carlo_precision.R"),
                  str(OUTPUT), str(INDEPENDENT_OUTPUT), sys.executable]
print(producer_command)
print(reader_command)
if RUN_PRECISION:
    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(producer_command, cwd=ROOT, check=True)
if RUN_INDEPENDENT_READBACK:
    INDEPENDENT_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(reader_command, cwd=ROOT, check=True)


In [ ]:
import json

if RUN_PRECISION:
    display(json.loads((OUTPUT / "result.json").read_text()))
if RUN_INDEPENDENT_READBACK:
    display(json.loads((INDEPENDENT_OUTPUT / "independent.json").read_text()))


## Scope

The candidate composition is conditional on valid constituent one-sided tests. It does not turn the earlier signed-effect power results into magnitude power. This notebook does not extract saved fits or reproduce the separate target-cell coverage diagnosis. The earlier conditional-statistics notebooks retain those populations and their original adverse coverage findings.

The independent precision reader uses R qbeta/dbinom, not the producer's numerical functions. It checks every scanned integer and count, as well as the input/source bindings. Self-consistent file hashes are not external archive authentication. Both components remain separate from any uncompleted population validation or human-study decision.